<a href="https://colab.research.google.com/github/342olive/scraped-data/blob/main/NLP-Applications/08_NLP_pipeline_for_kaggle_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile trainer_v2.py
# %load /kaggle/working/trainer_v2.py
# edited train.py
import config
import dataset
import engine
import torch
import pandas as pd
import torch.nn as nn
import numpy as np
import os

from model import BERTBaseUncased
from sklearn import model_selection
from sklearn import metrics

# Optimization - The New Way
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup


def train():
    # this function trains the model

    # read the training file and fill NaN values with "none"
    dfx = pd.read_csv(config.TRAINING_FILE).fillna("none")

    # sentiment = 1 if its positive else 0
    dfx.sentiment = dfx.sentiment.apply(
        lambda x: 1 if x == "positive" else 0
    )

    # split the data into training and validation
    df_train, df_valid = model_selection.train_test_split(
        dfx,
        test_size=0.1,
        random_state=42,
        stratify=dfx.sentiment.values
    )

    # reset index
    df_train = df_train.reset_index(drop=True)
    df_valid = df_valid.reset_index(drop=True)

    # training dataset
    train_dataset = dataset.BERTDataset(
        review=df_train.review.values,
        target=df_train.sentiment.values
    )

    # training dataloader
    train_data_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=config.TRAIN_BATCH_SIZE,
        num_workers=4
    )

    # validation dataset
    valid_dataset = dataset.BERTDataset(
        review=df_valid.review.values,
        target=df_valid.sentiment.values
    )

    # validation dataloader
    valid_data_loader = torch.utils.data.DataLoader(
        valid_dataset,
        batch_size=config.VALID_BATCH_SIZE,
        num_workers=1
    )

    # ADDED device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # model
    model = BERTBaseUncased()
    model.to(device)

    # optimizer parameters
    param_optimizer = list(model.named_parameters())
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]

    optimizer_parameters = [
        {
            "params": [
                p for n, p in param_optimizer
                if not any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.001,
        },
        {
            "params": [
                p for n, p in param_optimizer
                if any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]

    # number of training steps
    num_train_steps = int(
        len(df_train) / config.TRAIN_BATCH_SIZE * config.EPOCHS
    )

    # optimizer
    optimizer = AdamW(optimizer_parameters, lr=3e-5)

    # scheduler
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=num_train_steps
    )

    # DataParallel
    model = nn.DataParallel(model)

   # training loop
best_accuracy = 0.0
start_epoch = 0
checkpoint_path = "/kaggle/working/checkpoint.pt"

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    best_accuracy = checkpoint["best_accuracy"]
    start_epoch = checkpoint["epoch"] + 1

for epoch in range(start_epoch, config.EPOCHS):
    engine.train_fn(
        train_data_loader,
        model,
        optimizer,
        device,
        scheduler
    )

    outputs, targets = engine.eval_fn(
        valid_data_loader,
        model,
        device
    )

    outputs = np.array(outputs) >= 0.5
    accuracy = metrics.accuracy_score(targets, outputs)

    print(f"Accuracy Score = {accuracy}")

    if accuracy > best_accuracy:
        torch.save(model.state_dict(), config.MODEL_PATH)
        best_accuracy = accuracy

    # IMPORTANT: save checkpoint every epoch
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_accuracy": best_accuracy
    }, checkpoint_path)


if __name__ == "__main__":
    train()

